# Highway Network on CIFAR10

Assignment #1 — train a convolutional Highway Network (Srivastava et al., 2015) on CIFAR10.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine).

Run the cells top to bottom.

## 1. Get the code onto Colab

**Option A — git clone (recommended, if the project is on GitHub).**
Edit the URL to your repo, then run. If the files live in a subfolder, `cd` into it.

In [ ]:
# Option A: clone from GitHub
!git clone https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git
# If model.py etc. are in a subfolder, move into it (edit to match your repo layout):
# %cd <YOUR_REPO>/HighwayNetwork
!ls

In [ ]:
# Option B: manual upload (use ONLY if you are not cloning).
# Run this, then pick model.py, data.py, train.py, utils.py from your computer.
# from google.colab import files
# up = files.upload()

## 2. Check the GPU

In [ ]:
import torch
print('torch', torch.__version__)
print('GPU available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to GPU!')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 3. Imports + reproducible seed

In [ ]:
from model import HighwayConvNet
from data import get_dataloaders
from train import fit, evaluate
from utils import set_seed, count_parameters, plot_history, plot_comparison
import torch.nn as nn

set_seed(42)

## 4. Data (downloads CIFAR10, ~170 MB, first time only)

In [ ]:
train_loader, val_loader, test_loader, (mean, std) = get_dataloaders(batch_size=128, num_workers=2)
print('train-only mean:', mean.tolist())
print('train-only std: ', std.tolist())
print('batches  train:', len(train_loader), ' val:', len(val_loader), ' test:', len(test_loader))

## 5. Train the Highway network

`blocks_per_stage=3` → ~10 layers. On a T4 expect ~1–1.5 min/epoch, so ~40 epochs ≈ 40–60 min.
The best model (by val accuracy) is saved to `highway.pt`.

In [ ]:
highway = HighwayConvNet(mode='highway', blocks_per_stage=3, gate_bias=-2.0)
print('params:', f'{count_parameters(highway):,}')

hist_highway = fit(highway, train_loader, val_loader, device,
                   epochs=40, lr=0.1, weight_decay=5e-4, ckpt_path='highway.pt')

## 6. Training curves

In [ ]:
plot_history(hist_highway, title='Highway Network (10 layers)', save_path='highway_curves.png');

## 7. Final test accuracy (touch the test set ONCE)

Load the best saved model, then report test accuracy — this is your headline number.

In [ ]:
highway.load_state_dict(torch.load('highway.pt'))
test_loss, test_acc = evaluate(highway, test_loader, device, nn.CrossEntropyLoss())
print(f'TEST accuracy: {test_acc:.4f}   (loss {test_loss:.3f})')

## 8. (Day 4) Plain vs Highway — the paper's point

Train a **plain** net with the same architecture and compare. For the real effect,
re-run both at a larger depth (e.g. `blocks_per_stage=8`): plain should degrade, highway shouldn't.

In [ ]:
set_seed(42)
plain = HighwayConvNet(mode='plain', blocks_per_stage=3)
print('params:', f'{count_parameters(plain):,}')
hist_plain = fit(plain, train_loader, val_loader, device,
                 epochs=40, lr=0.1, weight_decay=5e-4, ckpt_path='plain.pt')

In [ ]:
plot_comparison({'highway': hist_highway, 'plain': hist_plain},
                metric='val_acc', title='Plain vs Highway — val accuracy',
                save_path='compare.png');